# Semantic Checkpoint Sweep

This notebook runs semantic segmentation testing across many Toy DINO and Graha checkpoints. For each checkpoint, it saves every test sample's model input tensor, hard prediction, target label, and metrics to disk.

The implementation lives in `semantic_checkpoint_sweep.py` so this notebook and the sbatch workflow use the same logic.

## Setup

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "semantic_checkpoint_sweep.py").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
for path in [REPO_ROOT, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from lfm.full_model.utils.utils import ensure_data_symlink
from semantic_checkpoint_sweep import build_config, discover_checkpoints, run_sweep

## Config

Set `TOY_CHECKPOINT_DIR` and `GRAHA_CHECKPOINT_DIR` after training finishes. Reminder: after rerunning Toy training, confirm the final checkpoint directory structure before launching the full 200-run sweep.

In [ ]:
# Data paths
INPUT_ROOT_DIR = None  # Optional source directory for ./data symlink
DATA_ROOT = None  # Leave as None to use notebooks/full_model/data
SIMLINK_DEST = INPUT_ROOT_DIR

# Checkpoint directories. Set these before running a real sweep.
BASE_DIR = Path("/explore/nobackup/people/ajkerr1/Lunar_FM")
TOY_CHECKPOINT_DIR = BASE_DIR / (
    "full_model_lfm/lfm/notebooks/full_model/outputs/"
    "toy_sem_seg_comparison/date_2026_07_17-time_14_48_43/checkpoints/toy_model"
)
GRAHA_CHECKPOINT_DIR = BASE_DIR / (
    "full_model_lfm/lfm/notebooks/full_model/outputs/"
    "toy_sem_seg_comparison/date_2026_07_17-time_14_48_43/checkpoints/full_model"
)
MODELS = ["toy", "graha"]  # Use ["toy"] or ["graha"] for one side only.

# Output root for checkpoint/sample outputs.
OUTPUT_ROOT = str(NOTEBOOK_DIR / "outputs" / "semantic_checkpoint_sweep")

# Data/model settings should match training.
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
TARGET_SIZE = 256
SPATIAL_TRANSFORM = "crop"
BATCH_SIZE = 16
NUM_WORKERS = 10  # Used once while preloading processed test batches.
NORMALIZE_INPUTS = True
MAX_TEST_SAMPLES = None  # Use a small number for smoke tests, e.g. 5.
MAX_CHECKPOINTS = None  # Use a small number for smoke tests, e.g. 1.

# Optional model/pretrain roots.
DINO_CHECKPOINT = None
GRAHA_PRETRAIN_DIR = None
GRAHA_STATS_BATCH_SIZE = 16
GRAHA_BATCH_SIZE = 16
GRAHA_NUM_WORKERS = 10
PRELOAD_TEST_BATCHES = True
SEED = 42
VERBOSE = False  # Set True to show datamodule/model setup printouts.

## Build Config

In [ ]:
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

args = Namespace(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    toy_checkpoint_dir=TOY_CHECKPOINT_DIR,
    graha_checkpoint_dir=GRAHA_CHECKPOINT_DIR,
    models=MODELS,
    band_filter=BAND_FILTER,
    target_size=TARGET_SIZE,
    spatial_transform=SPATIAL_TRANSFORM,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    normalize_inputs=NORMALIZE_INPUTS,
    max_test_samples=MAX_TEST_SAMPLES,
    dino_checkpoint=DINO_CHECKPOINT,
    graha_pretrain_dir=GRAHA_PRETRAIN_DIR,
    graha_stats_batch_size=GRAHA_STATS_BATCH_SIZE,
    graha_batch_size=GRAHA_BATCH_SIZE,
    graha_num_workers=GRAHA_NUM_WORKERS,
    max_checkpoints=MAX_CHECKPOINTS,
    seed=SEED,
    verbose=VERBOSE,
    preload_test_batches=PRELOAD_TEST_BATCHES,
)

config = build_config(args)
print("Data root:", config.data_root)
print("Output root:", config.output_root)
print("Models:", config.models)
print("Toy checkpoints:", config.toy_checkpoint_dir)
print("Graha checkpoints:", config.graha_checkpoint_dir)

## Inspect Checkpoints

Use this cell before running a large sweep to confirm checkpoint discovery and epoch parsing.

In [ ]:
if config.toy_checkpoint_dir is not None:
    toy_checkpoints = discover_checkpoints(config.toy_checkpoint_dir, max_checkpoints=config.max_checkpoints)
    print("Toy checkpoints:", len(toy_checkpoints))
    for item in toy_checkpoints[:5]:
        print(item.name, item.epoch, item.path)

if config.graha_checkpoint_dir is not None:
    graha_checkpoints = discover_checkpoints(config.graha_checkpoint_dir, max_checkpoints=config.max_checkpoints)
    print("Graha checkpoints:", len(graha_checkpoints))
    for item in graha_checkpoints[:5]:
        print(item.name, item.epoch, item.path)

## Run Sweep

For smoke tests, set `MAX_CHECKPOINTS = 1` and `MAX_TEST_SAMPLES = 2` in the config cell.

In [ ]:
results = run_sweep(config)
results.keys()